# Chapter 2, Preliminaries: Search for lower bounds on solutions to $F_t(X,Y) = \mu$

**Context:** This script computationally proves Lemma 2.2.3, which establishes that if $(x,y) \in \mathcal{O}_{L_t}^2$ is a non-trivial solution to $F_t(X,Y) = \mu$ with $\min\{|x|, |y|\} \leq 40$, then $t$ belongs to a finite, known set of quadratic integers. 

**Methodology:**
Because testing bounds up to 40 directly over quadratic integer fields is computationally prohibitive, this script implements some algebraic optimizations:
1. **Symmetry Reduction:** Reduces the search space by enforcing $\operatorname{Im}(x) > 0$ or $x > 0 \in \mathbb{Z}$ (Lemma 2.1).
2. **Norm Divisors:** When $\mu - x^4 \neq 0$, the script restricts the search for $y$ to values where $\operatorname{Norm}_{\mathbb{Q}(x)}(y)$ divides $\operatorname{Norm}_{\mathbb{Q}(x)}(\mu - x^4)$.

**Key Functions:**
* `get_x_candidates(d, m, M)`: Generates valid imaginary quadratic integers, applying symmetry relations to reduce the search space.
* `gen_solutions(d, x_ints, y0_vals)`: Uses norm divisors to find possible values for $y$ and solve for $t$.
* `low_y(m, M)`: The main runner that iterates through possible values of $t$ and outputs the final list of solutions.

In [ ]:
import itertools
from sage.all import *

def format_element(elt, d):
    """Helper function to format field elements as a + b*sqrt(-d)."""
    trace = elt.trace()
    a = trace / 2
    b = (elt - a) / sqrt(-d)
    return f"{a} + {b}*sqrt(-{d})"

def generate_d_list(m):
    """
    Generates a list of all squarefree integers d for which Q(sqrt(-d))
    contains some element with absolute value <= m (norm <= m^2).
    """
    dlist = []
    max_norm = floor(4 * m**2 - 1)
    
    for d in range(1, max_norm + 1):
        if not Integer(d).is_squarefree():
            continue
            
        rem = d % 4
        if rem in (1, 2) and d <= m**2:
            dlist.append(d)
        elif rem == 3:
            dlist.append(d)
            
    return dlist

def get_x_candidates(d, m, M):
    """
    Returns imaginary quadratic integers with absolute value between m and M 
    in Q(sqrt(-d)), applying symmetry relations to reduce the search space.
    """
    x_candidates = set()
    L = NumberField(x**2 + d, 'rootd')
    rootd = L.gen()
    
    for i in range(floor(m**2), floor(M**2) + 1):
        elts = L.elements_of_norm(i)
        
        # Ensure we capture all associates
        if Integer(i).is_square():
            iroot = L(sqrt(i))
            for u in L.roots_of_unity():
                elts.append(iroot * u)
                
        for j in elts:
            a = QQ(j.trace() / 2)
            b = QQ((j - a) / rootd)
            # Enforce symmetry (restricting to positive real/imaginary parts)
            x_candidates.add((sgn(a) * a, b))
            
    return x_candidates

def all_quadratic_integers_x(dlist, m, M):
    """Returns a dictionary mapping d to its x-candidates."""
    return {d: get_x_candidates(d, m, M) for d in dlist}

def gen_solutions(d, x_ints, y0_vals):
    """
    Finds and returns all valid t for non-trivial solutions (x, y) to Ft(x,y) = mu.
    Returns a set of formatted solution strings.
    """
    K = QuadraticField(-d, 'rootdi')
    rootdi = K.gen()
    rofu = K.roots_of_unity()
    
    found_solutions = set()
    
    # Iterate over all pairs (mu, x)
    for mu, x_tup in itertools.product(rofu, x_ints):
        x1 = K(x_tup[0] + x_tup[1] * rootdi)
        if x1 == 0:
            continue
            
        mu_minus_x4 = mu - x1**4
        norm_mu_x4 = mu_minus_x4.norm()
        
        # Branch 1: mu - x^4 = 0
        if norm_mu_x4 == 0:
            for y_tup in y0_vals:
                y0 = K(y_tup[0] + y_tup[1] * rootdi)
                if y0 == 0:
                    continue
                    
                RS = 6 * x1**2 * y0 - y0**3
                LS = x1 * y0**2 - x1**3
                
                if RS == 0 and LS == 0:
                    found_solutions.add(f"x = {format_element(x1, d)}, y = {format_element(y0, d)} | t = all t")
                elif LS != 0 and (RS / LS).is_integral():
                    t_val = RS / LS
                    found_solutions.add(f"x = {format_element(x1, d)}, y = {format_element(y0, d)} | t = {t_val}")
            continue
            
        # Branch 2: mu - x^4 != 0 (Using Norm Divisors)
        divs = divisors(norm_mu_x4)
        
        for n in divs:
            base_y_elts = K.elements_of_norm(n)
            y_candidates = set()
            
            # Apply symmetry reduction
            for base_y in base_y_elts:
                for u in rofu:
                    y_temp = base_y * u
                    a = QQ(y_temp.trace() / 2)
                    b = QQ((y_temp - a) / rootdi)
                    y_candidates.add(K((sgn(a) * a) + b * rootdi))
            
            for y1 in y_candidates:
                if y1 == 0:
                    continue
                    
                if not (mu_minus_x4 / y1).is_integral():
                    continue
                    
                RS = (mu_minus_x4 / y1) + 6 * x1**2 * y1 - y1**3
                LS = x1 * y1**2 - x1**3
                
                if RS == 0 and LS == 0:
                    found_solutions.add(f"x = {format_element(x1, d)}, y = {format_element(y1, d)} | t = all t")
                elif LS != 0:
                    t_val = RS / LS
                    # Check for integrality and exclude +/- 4i
                    if t_val.is_integral() and t_val not in ZZ and t_val not in (4*rootdi, -4*rootdi):
                        found_solutions.add(f"x = {format_element(x1, d)}, y = {format_element(y1, d)} | t = {t_val}")
                        
    return sorted(list(found_solutions))

def low_y(m, M):
    """Main runner: Finds all solutions x,y with min{|x|,|y|} between m and M."""
    dlist = generate_d_list(M)
    total_d = len(dlist)
    
    # Precompute all x candidates and the small y-bound checks
    dict_x_ints = all_quadratic_integers_x(dlist, m, M)
    dict_y_norm0 = all_quadratic_integers_x(dlist, 0, 7) # y <= 7 bounds
    
    # Dictionary to store all found solutions for a final summary if needed
    all_results = {}
    
    for i, d in enumerate(dlist):
        # Print the progress bar
        print(f"\rScanning d = {d} (Progress: {i + 1}/{total_d})...", end="", flush=True)
        
        x_cands = dict_x_ints[d]
        y_bounds = dict_y_norm0[d]
        
        # Collect the solutions instead of printing them inline
        sols = gen_solutions(d, x_cands, y_bounds)
        
        if sols:
            # Clear the progress bar line by overwriting with spaces, then drop down
            print(f"\r{' ' * 50}\r", end="") 
            print(f"=== Solutions in Q(sqrt(-{d})) ===")
            for sol in sols:
                print(f"  • {sol}")
            print() # Add a blank line for readability
            
            all_results[d] = sols
            
    print(f"\rSearch complete!{' ' * 30}")
    return all_results

In [14]:
%time low_y(0,10)

=== Solutions in Q(sqrt(-1)) ===                  
  • x = 0 + -1*sqrt(-1), y = 1 + -1*sqrt(-1) | t = -4*rootdi
  • x = 0 + -1*sqrt(-1), y = 2 + 0*sqrt(-1) | t = -4*rootdi
  • x = 0 + 1*sqrt(-1), y = 1 + -1*sqrt(-1) | t = 4*rootdi
  • x = 0 + 1*sqrt(-1), y = 2 + 0*sqrt(-1) | t = 4*rootdi
  • x = 1 + 0*sqrt(-1), y = 0 + -2*sqrt(-1) | t = 4*rootdi
  • x = 1 + 0*sqrt(-1), y = 0 + 2*sqrt(-1) | t = -4*rootdi
  • x = 1 + 0*sqrt(-1), y = 1 + -1*sqrt(-1) | t = 4*rootdi

=== Solutions in Q(sqrt(-2)) ===                  
  • x = 0 + 1/2*I*sqrt(2)*sqrt(-2)*sqrt(-2), y = 1 + 0*sqrt(-2) | t = -3*rootdi
  • x = 1 + 0*sqrt(-2), y = 0 + -1/2*I*sqrt(2)*sqrt(-2)*sqrt(-2) | t = -3*rootdi
  • x = 1 + 0*sqrt(-2), y = 0 + 1/2*I*sqrt(2)*sqrt(-2)*sqrt(-2) | t = 3*rootdi

=== Solutions in Q(sqrt(-3)) ===                  
  • x = 1 + 0*sqrt(-3), y = 1/2 + -1/6*I*sqrt(3)*sqrt(-3)*sqrt(-3) | t = -2*rootdi
  • x = 1 + 0*sqrt(-3), y = 1/2 + -1/6*I*sqrt(3)*sqrt(-3)*sqrt(-3) | t = -5/2*rootdi + 1/2
  • x = 1 + 0*sq

{1: ['x = 0 + -1*sqrt(-1), y = 1 + -1*sqrt(-1) | t = -4*rootdi',
  'x = 0 + -1*sqrt(-1), y = 2 + 0*sqrt(-1) | t = -4*rootdi',
  'x = 0 + 1*sqrt(-1), y = 1 + -1*sqrt(-1) | t = 4*rootdi',
  'x = 0 + 1*sqrt(-1), y = 2 + 0*sqrt(-1) | t = 4*rootdi',
  'x = 1 + 0*sqrt(-1), y = 0 + -2*sqrt(-1) | t = 4*rootdi',
  'x = 1 + 0*sqrt(-1), y = 0 + 2*sqrt(-1) | t = -4*rootdi',
  'x = 1 + 0*sqrt(-1), y = 1 + -1*sqrt(-1) | t = 4*rootdi'],
 2: ['x = 0 + 1/2*I*sqrt(2)*sqrt(-2)*sqrt(-2), y = 1 + 0*sqrt(-2) | t = -3*rootdi',
  'x = 1 + 0*sqrt(-2), y = 0 + -1/2*I*sqrt(2)*sqrt(-2)*sqrt(-2) | t = -3*rootdi',
  'x = 1 + 0*sqrt(-2), y = 0 + 1/2*I*sqrt(2)*sqrt(-2)*sqrt(-2) | t = 3*rootdi'],
 3: ['x = 1 + 0*sqrt(-3), y = 1/2 + -1/6*I*sqrt(3)*sqrt(-3)*sqrt(-3) | t = -2*rootdi',
  'x = 1 + 0*sqrt(-3), y = 1/2 + -1/6*I*sqrt(3)*sqrt(-3)*sqrt(-3) | t = -5/2*rootdi + 1/2',
  'x = 1 + 0*sqrt(-3), y = 1/2 + -1/6*I*sqrt(3)*sqrt(-3)*sqrt(-3) | t = -5/2*rootdi - 1/2',
  'x = 1 + 0*sqrt(-3), y = 1/2 + 1/6*I*sqrt(3)*sqrt(-3)*